In [4]:
import os
import numpy as np
import torch

from cosmos_predict2.utils.printer import print_batch
from cosmos_predict2.utils.vis_helpers import save_action_as_image
from imaginaire.utils.io import save_image_or_video

from imaginaire.lazy_config import LazyCall as L
from cosmos_predict2.data.action_conditioned.uha_dataset import OxeUhaDataModule, NoEncoder

In [12]:
transforms_conf = dict(
    move_axis=True,
    bytes_to_string=True,
    adjust_type=None,
    add_robot_information=False
)

language_encoders_conf = dict(
    model_name=None
)

frame_transform_kwargs=dict(
    image_augment_kwargs=dict(
        primary=dict(
            random_resized_crop=dict(
                scale=(0.8, 1.0),
                ratio=(0.9, 1.1)
            ),
            random_brightness=(0.1,),
            random_contrast=(0.9, 1.1),
            random_saturation=(0.9, 1.1),
            random_hue=(0.05,),
            augment_order=(
                "random_resized_crop",
                "random_brightness",
                "random_contrast",
                "random_saturation",
                "random_hue",
            ),
        ),
        secondary=dict(
            random_resized_crop=dict(
                scale=(0.8, 1.0),
                ratio=(0.9, 1.1)
            ),
            random_brightness=(0.1,),
            random_contrast=(0.9, 1.1),
            random_saturation=(0.9, 1.1),
            random_hue=(0.05,),
            augment_order=(
                "random_resized_crop",
                "random_brightness",
                "random_contrast",
                "random_saturation",
                "random_hue",
            ),
        ),
        wrist=dict(
            random_brightness=(0.1,),
            random_contrast=(0.9, 1.1),
            random_saturation=(0.9, 1.1),
            random_hue=(0.05,),
            augment_order=(
                "random_brightness",
                "random_contrast",
                "random_saturation",
                "random_hue",
            ),
        ),
    ),
    resize_size=dict(
        primary=(176, 176),
        secondary=(160, 160),  # not used
        wrist=(84, 84),  # all black
    ),
    resize_size_future_obs=dict(
        primary=(176, 176),
        secondary=(160, 160),  # should be same as resize_size
        wrist=(84, 84),
    ),
    num_parallel_calls=6,
)


DEBUG_DATASET = "fractal"
DEBUG_DATASET_MAPPING = {
    "fractal": "fractal",
    "bridge2": "bridge",
}

n_v_cond, n_v_out = 4 * 1 + 1, 4 * 5  # 4+1+20=25
n_a_out = n_v_out
n_latent_v_cond, n_latent_v_out = 1 * 1 + 1, 1 * 5  # 1+1+5=7
horizon = n_v_cond + n_v_out # 25
pad_before = n_v_cond - 1
datasets_conf = dict(
    DATA_NAME=DEBUG_DATASET_MAPPING[DEBUG_DATASET],
    DATA_PATH="/home/geyuan/local_soft/huggingface/v1/",
    load_camera_views=["primary", "secondary", "wrist"],  # ori: ["primary", "secondary", "wrist"],
    load_proprio=True,  # ori: False
    load_language_embeddings=True,  # ori: False
    action_proprio_normalization_type="bounds",
    interleaved_dataset_cfg=dict(
        shuffle_buffer_size=5000,  # ori: 400000
        balance_weights=True,
        traj_transform_kwargs=dict(
            goal_relabeling_strategy=None,
            goal_relabeling_kwargs=dict(
                min_bound=20,
                max_bound=50,
                frame_diff=3
            ),
            window_size=n_v_cond,
            action_horizon=n_a_out,
            skip_unlabeled=True,
            load_future_frames=True, # NOTE: ori: False
        ),
        frame_transform_kwargs=frame_transform_kwargs,
        traj_transform_threads=16,
        traj_read_threads=8,
    )
)

uha_datamodule = OxeUhaDataModule(
    transforms=transforms_conf,
    language_encoders=language_encoders_conf,
    datasets=datasets_conf,
    batch_size=20 * 8,
    drop_last=True,
    # CosmosPredict2 specific
    use_ori_uha_data_collate=False,
    p_camera_drop=0.,
    p_proprio_drop=0.,
    state_t=n_latent_v_cond + n_latent_v_out,
)
# uha_datamodule.prepare_data()
# uha_datamodule.setup()

train_dataloader = uha_datamodule.train_dataloader()

[DEBUG] OxeUhaDataModule prepare_data finished.
[DEBUG] Loading language embeddings from: /home/geyuan/local_soft/huggingface/v1/lang_emb_t5xxl/fractal/t5_embeddings.npz
[DEBUG] Created static lookup with 599 embeddings, shape: (512, 1024)


2025-10-10 10:01:59.016441: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization
2025-10-10 10:01:59.990669: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization



######################################################################################
# Loading the following 1 datasets (incl. sampling weight):                         #
# fractal20220817_data: ====================================================1.000000 #
######################################################################################

[DEBUG] Loading language embeddings from: /home/geyuan/local_soft/huggingface/v1/lang_emb_t5xxl/fractal/t5_embeddings.npz
[DEBUG] Created static lookup with 599 embeddings, shape: (512, 1024)


2025-10-10 10:02:04.301359: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


[DEBUG] Action shape before transforms: TensorSpec(shape=(None, 7), dtype=tf.float32, name=None)
[DEBUG] Action shape after transforms: TensorSpec(shape=(5, 20, 7), dtype=tf.float32, name=None)
[DEBUG] OxeUhaDataModule setup finished (main=True). Train len=23665, Info: {'train_dataset': {'fractal20220817_data': {'action': {'mean': array([ 0.00698757,  0.00626593, -0.01262515,  0.04333356, -0.00575619,
        0.00091303,  0.53542048]), 'std': array([0.0692113 , 0.05970518, 0.07353121, 0.15610436, 0.13164447,
       0.1459381 , 0.49711195]), 'max': array([ 2.99845934, 22.09052849,  2.75075245,  1.57063651,  1.53210866,
        1.56915224,  1.        ]), 'min': array([-2.02045202, -5.49789953, -2.03166342, -1.56991792, -1.56989217,
       -1.57041943,  0.        ]), 'p99': array([0.17824687, 0.1493838 , 0.21842355, 0.5892666 , 0.35272657,
       0.44796681, 1.        ]), 'p01': array([-0.22453528, -0.14820013, -0.23158971, -0.35179949, -0.41930113,
       -0.43643461,  0.        ]), 'mas

In [13]:
from tqdm import tqdm

vis_idx = 0
in_sample = None

for idx, batch in enumerate(tqdm(train_dataloader)):
    if idx == 0:
        print_batch(DEBUG_DATASET, batch)
    if idx < vis_idx:
        continue
    print_batch(DEBUG_DATASET, batch)
    # print("language:", batch["task"]["language_instruction"])

    in_sample = batch
    break

  0%|                                                                                               | 0/23665 [00:00<?, ?it/s]W0000 00:00:1760061735.470010 3921528 op_level_cost_estimator.cc:699] Error in PredictCost() for the op: op: "CropAndResize" attr { key: "T" value { type: DT_FLOAT } } attr { key: "extrapolation_value" value { f: 0 } } attr { key: "method" value { s: "bilinear" } } inputs { dtype: DT_FLOAT shape { dim { size: 1 } dim { size: 176 } dim { size: 176 } dim { size: -256 } } } inputs { dtype: DT_FLOAT shape { dim { size: -2 } dim { size: 4 } } } inputs { dtype: DT_INT32 shape { dim { size: -2 } } } inputs { dtype: DT_INT32 shape { dim { size: 2 } } } device { type: "CPU" vendor: "GenuineIntel" model: "111" frequency: 2100 num_cores: 192 environment { key: "cpu_instruction_set" value: "AVX SSE, SSE2, SSE3, SSSE3, SSE4.1, SSE4.2" } environment { key: "eigen" value: "3.4.90" } l1_cache_size: 49152 l2_cache_size: 2097152 l3_cache_size: 110100480 memory_size: 268435456 } o

RuntimeError: Sizes of tensors must match except in dimension 2. Expected size 176 but got size 160 for tensor number 1 in the list.

In [11]:
""" Visualization Remapped Dataloader """
max_vis_len = 50
mv_sample = in_sample


horizon = mv_sample['action'][0].shape[0]
save_image_or_video(
    mv_sample['video'][2, :horizon].float() / 255.,
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentview.mp4",
    fps=4
)

if DEBUG_DATASET == "fractal":
    meta_p01 = [-0.22453528, -0.14820013, -0.23158971, -0.35179949, -0.41930113, -0.43643461,  0.        ]
    meta_p99 = [0.17824687, 0.1493838 , 0.21842355, 0.5892666 , 0.35272657, 0.44796681, 1.        ]
elif DEBUG_DATASET == "bridge2":
    meta_p01 = [-0.02853955, -0.04143204, -0.02597738, -0.08020887, -0.0921306 , -0.20548619,  0.        ]
    meta_p99 = [0.02812228, 0.04063032, 0.03994889, 0.08121916, 0.07724379, 0.2021405 , 1.        ]

def denorm_action(act):
    return (act + 1) / 2 * (np.array(meta_p99) - np.array(meta_p01)) + np.array(meta_p01)

save_action_as_image(
    denorm_action(mv_sample['action'])[:, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action012.png",
)
save_action_as_image(
    denorm_action(mv_sample['action'])[:, 6:7],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_action6.png",
)

save_action_as_image(
    mv_sample['agent_pos'][:, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos012.png",
)
# save_action_as_image(
#     mv_sample['agent_pos'][:, -1:] * dataset_multi_view.meta_gripper_states_std[-1] + dataset_multi_view.meta_gripper_states_mean[-1],
#     "/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_mv_agentpos6.png",
# )


KeyError: 'video'

In [5]:
""" Visualization Original Dataloader """
max_vis_len = 50

def save_view(in_sample_, batch_key_: str, view_key_: str):
    in_video_ = in_sample_[batch_key_][f'image_{view_key_}']  # (B,T,C,H,W)
    in_video_ = in_video_.permute(0, 2, 1, 3, 4)  # (B,C,T,H,W)
    suffix = "_future" if "future" in batch_key_ else ""
    save_image_or_video(
        in_video_[:, :max_vis_len].to(torch.float32) / 255.,  # (c,t,h,w)
        f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_{view_key_}{suffix}.mp4",
        fps=10
    )

# 1. Views
save_view(in_sample, 'observation', 'primary')
save_view(in_sample, 'observation', 'secondary')
save_view(in_sample, 'observation', 'wrist')

save_view(in_sample, 'future_frames', 'primary')
save_view(in_sample, 'future_frames', 'secondary')
save_view(in_sample, 'future_frames', 'wrist')

# 2. Actions
in_action = in_sample['action'][:, -1]  # (B,T,H,D) -> (B,H,D), in [-1,1]
save_action_as_image(
    in_action[0, :max_vis_len, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_normed_action.png",
)

if DEBUG_DATASET == "fractal":
    meta_p01 = [-0.22453528, -0.14820013, -0.23158971, -0.35179949, -0.41930113, -0.43643461,  0.        ]
    meta_p99 = [0.17824687, 0.1493838 , 0.21842355, 0.5892666 , 0.35272657, 0.44796681, 1.        ]
elif DEBUG_DATASET == "bridge2":
    meta_p01 = [-0.02853955, -0.04143204, -0.02597738, -0.08020887, -0.0921306 , -0.20548619,  0.        ]
    meta_p99 = [0.02812228, 0.04063032, 0.03994889, 0.08121916, 0.07724379, 0.2021405 , 1.        ]
in_action_unnormed = (in_action + 1) / 2 * (np.array(meta_p99) - np.array(meta_p01)) + np.array(meta_p01)
save_action_as_image(
    in_action_unnormed[0, :max_vis_len, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_unnormed_action.png",
)

# 3. Language
in_language = in_sample['task']['language_instruction'][0]
in_lang_emb = in_sample['task']['language_embedding'][0]
print(in_language)
print(in_lang_emb[0, :10])

data = np.load(f"/home/geyuan/local_soft/huggingface/v1/lang_emb_t5xxl/{DEBUG_DATASET}/t5_embeddings.npz", allow_pickle=True)
map_text_2_emb = data['text_to_embedding_map'].item()
print(map_text_2_emb[in_language][0, :10])

# 4. Proprio
in_proprio = in_sample['observation']['proprio']  # (B,T,D), in [-1,1]
dataset_meta = uha_datamodule.dataset_info
subdataset_meta = list(dataset_meta['train_dataset'].values())[0]
meta_proprio_p01 = subdataset_meta['proprio']['p01']
meta_proprio_p99 = subdataset_meta['proprio']['p99']
in_proprio_unnormed = (in_proprio + 1) / 2 * (np.array(meta_proprio_p99) - np.array(meta_proprio_p01)) + np.array(meta_proprio_p01)
save_action_as_image(
    in_proprio_unnormed[0, :max_vis_len, :3],
    f"/home/geyuan/code/cospred2nvidia/output/de_{DEBUG_DATASET}_in_unnormed_proprio.png",
)


KeyError: 'observation'